# Phase 3 — Clinically gated fusion (Model D)

## The hypothesis

Concatenation (Model C) lets the head add clinical evidence to image evidence. It cannot express
the thing we actually believe: that clinical measurements change **how the scan should be read**.
A high CST means macular thickening, which should raise the weight the model puts on
fluid-related image features — a *multiplicative* interaction, not an additive one.

```
OCT   -> image encoder    -> image embedding ---------+
                                                       -> gated fusion -> multilabel head
BCVA/CST -> clinical encoder -> clinical embedding ---+
```

with

```
gate   = sigmoid(MLP(clinical_embedding))
scale  = 1 + alpha * (2 * gate - 1)      # residual: 1.0 means no modulation
gated  = image_embedding * scale
fused  = concat(gated, clinical_embedding)
```

## The failure mode this is built to avoid

A gate initialised carelessly collapses toward zero and erases the OCT signal before the image
encoder has learned anything. `ClinicalGate` zeroes its output weights and sets the bias so the
scale starts at **exactly 1.0** — a pure pass-through the model must learn to move away from.

## The question that decides whether this worked

Beating Model C on a metric is not enough. If the gate stayed at identity, or collapsed to a
constant the head could have absorbed into a bias, then Model D *is* Model C with extra
parameters and any difference is noise. Section 5 tests that directly.

---
## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, warnings, time
from pathlib import Path

REPO_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
    Path.cwd(),
)
sys.path.insert(0, str(REPO_ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

from olives_biomarkers import (
    ExperimentRunner, ExperimentSuite, OlivesPipeline, ResultsAggregator, RunResult,
)
from olives_biomarkers.config import ConfigLoader
from olives_biomarkers.evaluation import GateAnalyzer, ResultsPlotter

pipeline = OlivesPipeline.from_config(REPO_ROOT / "configs" / "data.yaml", repo_root=REPO_ROOT)
loader = ConfigLoader(REPO_ROOT)

BUDGET = "colab_gpu" if pipeline.env.device == "cuda" else "local_cpu"
budget = loader.load(REPO_ROOT / "configs" / f"{BUDGET}.yaml")
RUNS_DIR = REPO_ROOT / "outputs" / "runs" / BUDGET
FIGURES = REPO_ROOT / "outputs" / "figures" / "phase3"
FIGURES.mkdir(parents=True, exist_ok=True)

plotter = ResultsPlotter()

def show(figure, name):
    if figure is None:
        print(f"(no figure for {name})")
        return
    plotter.save(figure, FIGURES / f"{name}.png")
    plt.show()

print(f"device={pipeline.env.device}  budget={BUDGET}  runs={RUNS_DIR}")

In [ ]:
manifest = pipeline.get_manifest()
frame = pipeline.modelling_frame(manifest, attach_cache=True)
assignment = pipeline.make_holdout_split(manifest, write=False)   # identical to Phase 2
n_labels = len(manifest.label_columns)

print(f"{len(frame):,} scans | {frame.patient_id.nunique()} patients | {n_labels} labels")
print({k: len(v) for k, v in assignment.partitions.items()})

---
## 2. The gate at initialisation

Before training anything, confirm the safety property holds: the applied scale must be 1.0 for
every sample and every channel, so the model starts as an exact copy of the ungated path.

In [ ]:
from olives_biomarkers.models import ModelFactory

cfg_gated = loader.load(REPO_ROOT / "configs" / "fusion_gated.yaml")
probe = ModelFactory().build(cfg_gated.model, n_labels=n_labels, clinical_dim=4)

stats = probe.gate_statistics(torch.randn(256, 4))
display(pd.Series(stats).to_frame("value").round(5))

assert abs(stats["scale_mean"] - 1.0) < 0.02, "gate is not an identity at init"
assert stats["scale_std"] < 1e-4, "gate varies at init; it should be a constant pass-through"
print("\nOK - the gate starts as an exact identity, so the OCT signal cannot be erased "
      "before training begins.")

In [ ]:
# And the gradient still flows, so the zeroed initialisation is not a dead end.
from olives_biomarkers.training.losses import MaskedBCEWithLogitsLoss

probe.train()
loss = MaskedBCEWithLogitsLoss()(
    probe(image=torch.randn(4, 3, 64, 64), clinical=torch.randn(4, 4)),
    torch.randint(0, 2, (4, n_labels)).float(),
)
loss.backward()
gate_grad = probe.gate.projection[-1].weight.grad
print(f"gate output-layer gradient magnitude: {gate_grad.abs().sum():.6f}")
assert gate_grad.abs().sum() > 0, "no gradient reaches the gate - it can never learn"
print("OK - the gate can learn away from identity.")

---
## 3. Training Model D

Same split, same budget, same seeds as Phase 2. Nothing else may differ, or the comparison is
not a comparison.

In [ ]:
def make_config(stem):
    cfg = loader.load(REPO_ROOT / "configs" / f"{stem}.yaml")
    cfg.data.image_size = budget.data.image_size
    cfg.data.num_workers = budget.data.num_workers
    cfg.training.epochs = budget.training.epochs
    cfg.training.batch_size = budget.training.batch_size
    cfg.training.learning_rate = budget.training.learning_rate
    cfg.training.early_stopping_patience = budget.training.early_stopping_patience
    cfg.training.amp = budget.training.amp
    return cfg

SEEDS = [42] if BUDGET == "local_cpu" else [42, 43, 44]
RUN_TRAINING = True    # False -> load the runs saved by scripts/run_comparison.py

if RUN_TRAINING:
    suite = ExperimentSuite(pipeline, output_root=RUNS_DIR)
    gated_results = suite.run_models(
        configs={"gated_fusion": make_config("fusion_gated")},
        assignment=assignment,
        manifest=manifest,
        seeds=SEEDS,
    )
else:
    gated_results = [
        r for r in RunResult.load_all(RUNS_DIR) if r.model_name == "gated_fusion"
    ]
print(f"{len(gated_results)} gated-fusion run(s)")

In [ ]:
# Bring in the Phase 2 baselines for the comparison.
all_results = RunResult.load_all(RUNS_DIR)
print(f"{len(all_results)} runs total: {sorted({r.model_name for r in all_results})}")

aggregator = ResultsAggregator(all_results)
comparison = aggregator.comparison(sort_by="macro_auprc")
comparison[["model", "seed", "n_parameters", "epochs_run", "minutes",
            "macro_f1", "macro_auroc", "macro_auprc"]]

In [ ]:
show(plotter.model_comparison(comparison), "01_all_models")

---
## 4. Does D beat C?

The comparison that matters is gated fusion against **concatenation**, not against the OCT
baseline: both C and D see the clinical features, so C is what isolates the contribution of
*gating* specifically.

In [ ]:
means = comparison.groupby("model")[["macro_f1", "macro_auroc", "macro_auprc"]].mean()
display(means.round(4))

if {"concat_fusion", "gated_fusion"}.issubset(means.index):
    delta = means.loc["gated_fusion"] - means.loc["concat_fusion"]
    print("\nGated fusion minus concatenation:")
    for metric, value in delta.items():
        print(f"  {metric:>14}: {value:+.4f}")
    print("\nParameter cost of the gate:")
    params = comparison.groupby("model")["n_parameters"].first()
    extra = params.get("gated_fusion", 0) - params.get("concat_fusion", 0)
    print(f"  {extra:+,} parameters ({100 * extra / params.get('concat_fusion', 1):+.1f}%)")
    print("\nA point-estimate gap is not yet a result. Phase 5 checks whether the patient-level")
    print("confidence intervals actually separate.")

In [ ]:
histories = {
    r.model_name: r.history.to_frame()
    for r in all_results
    if r.seed == SEEDS[0] and len(r.history.records)
}
show(plotter.training_curves(histories, monitor="val_macro_auprc"), "02_training_curves")

---
## 5. What did the gate actually learn?

This is the section that decides whether Model D is a real mechanism or an expensive
re-parameterisation of Model C.

Three checks:

1. **Did it move?** Mean `|scale - 1|` near zero means the gate never left identity.
2. **Does it respond to input?** If per-channel standard deviation across samples is ~0, the gate
   is a learned constant that the classification head could have absorbed into its bias.
3. **Does it respond to the *right* input?** The gate should correlate with BCVA/CST, and should
   separate the fluid biomarkers more than the vitreous-face ones.

In [ ]:
runner = ExperimentRunner(pipeline, make_config("fusion_gated"), output_root=RUNS_DIR)
data = runner.build_data_module(frame, assignment, manifest.label_columns)

gated_run = next((r for r in all_results if r.model_name == "gated_fusion"), None)
assert gated_run is not None, "no gated-fusion run found; train one first"

model = ModelFactory().build(
    make_config("fusion_gated").model, n_labels=n_labels, clinical_dim=data.preprocessor.output_dim
)
from olives_biomarkers.training.callbacks import CheckpointManager

CheckpointManager(gated_run.run_dir / "checkpoints", run_id=gated_run.run_id).load(
    model, map_location=pipeline.env.device
)
print(f"loaded trained weights from {gated_run.run_id}")

In [ ]:
analyzer = GateAnalyzer(model, device=pipeline.env.device)
collected = analyzer.collect(data.dataloader("test", shuffle=False))
summary = analyzer.summary(collected)
display(pd.Series(summary).to_frame("value").round(5))

print("\nInterpretation:")
notes = analyzer.interpret(summary)
if notes:
    for note in notes:
        print(f"  - {note}")
else:
    print("  - The gate moved off identity and varies with input, without collapsing.")

In [ ]:
show(plotter.gate_distribution(collected), "03_gate_distribution")

In [ ]:
# Does the gate respond to the clinical features it is conditioned on?
feature_names = data.preprocessor.feature_names
display(analyzer.clinical_response(collected, feature_names))
print("A gate that is genuinely clinically driven shows non-trivial correlation with at least")
print("one clinical feature. Near-zero everywhere means it is conditioning on nothing.")

In [ ]:
# Does gating separate the biomarkers CST should inform?
gate_by_label = analyzer.gate_by_label(collected, manifest.label_columns)
display(gate_by_label)
print("EDA predicted CST tracks the fluid biomarkers (irf, drt_me, srf). If the gate is doing")
print("clinically sensible work, those should sit near the top of this table rather than")
print("pavf/favf, which concern structure above the retina that CST does not measure.")

In [ ]:
display(analyzer.channel_activity(collected, top_n=15))

---
## 6. Ablations

Two questions the headline number cannot answer.

In [ ]:
# Ablation E: does the missingness indicator matter?
# BCVA/CST are missing for exactly one patient, so the indicator is highly informative
# in principle - but it may simply let the model memorise that patient.
RUN_ABLATIONS = True

ablations = {}
if RUN_ABLATIONS:
    cfg_no_missing = make_config("fusion_gated")
    cfg_no_missing.model.use_missingness_indicators = False
    cfg_no_missing.experiment.name = "gated_no_missingness"
    ablations["gated_no_missingness"] = cfg_no_missing

    # Ablation: non-residual gating (raw multiplicative gate).
    cfg_raw_gate = make_config("fusion_gated")
    cfg_raw_gate.model.gate_residual = False
    cfg_raw_gate.model.gate_bias_init = None      # identity-preserving for this mode
    cfg_raw_gate.experiment.name = "gated_raw_multiplicative"
    ablations["gated_raw_multiplicative"] = cfg_raw_gate

    suite = ExperimentSuite(pipeline, output_root=RUNS_DIR / "ablations")
    ablation_results = suite.run_models(
        configs=ablations,
        assignment=assignment,
        manifest=manifest,
        seeds=[SEEDS[0]],
    )
else:
    ablation_results = RunResult.load_all(RUNS_DIR / "ablations")

print(f"{len(ablation_results)} ablation run(s)")

In [ ]:
combined = ResultsAggregator(all_results + ablation_results)
table = combined.comparison(sort_by="macro_auprc")
table[["model", "experiment", "seed", "macro_f1", "macro_auroc", "macro_auprc"]]

---
## 7. Per-label view

Where, specifically, does gating change anything? A gate justified by CST should help the fluid
biomarkers and leave the vitreous-face ones alone. Broad, uniform gains are more likely to be
seed noise than mechanism.

In [ ]:
pivot = aggregator.per_label_pivot(metric="auprc")
display(pivot)
show(plotter.per_label_comparison(pivot, metric="auprc"), "04_per_label_auprc")

In [ ]:
if {"concat_fusion", "gated_fusion"}.issubset(pivot.columns):
    delta = (pivot["gated_fusion"] - pivot["concat_fusion"]).sort_values(ascending=False)
    frame_delta = delta.to_frame("auprc_gain_over_concat")
    frame_delta["n_positive"] = pivot["n_positive"]
    display(frame_delta)

    fluid = [c for c in ["irf", "drt_me", "srf"] if c in delta.index]
    vitreous = [c for c in ["pavf", "favf", "vitreous_debris"] if c in delta.index]
    if fluid and vitreous:
        print(f"\nmean gain on fluid biomarkers    {delta[fluid].mean():+.4f}")
        print(f"mean gain on vitreous biomarkers {delta[vitreous].mean():+.4f}")
        print("\nThe mechanistic story predicts the first is larger. If they are similar, the")
        print("gate is not doing what the hypothesis says it does, whatever the macro metric says.")

---
## 8. Findings

Fill this in from the numbers above rather than from expectation.

In [ ]:
print("PHASE 3 SUMMARY")
print("=" * 60)
print(f"Runs compared : {sorted({r.model_name for r in all_results})}")
print(f"Seeds         : {SEEDS}")
print(f"Budget        : {BUDGET} ({budget.training.epochs} epochs, {budget.data.image_size})")
print()
if {"concat_fusion", "gated_fusion"}.issubset(means.index):
    gain = means.loc["gated_fusion", "macro_auprc"] - means.loc["concat_fusion", "macro_auprc"]
    print(f"macro AUPRC, gated - concat : {gain:+.4f}")
print(f"gate moved from identity by : {summary['mean_absolute_deviation_from_identity']:.4f}")
print(f"gate channels ~constant      : {100 * summary['fraction_channels_essentially_constant']:.0f}%")
print()
print("Verdict is only supportable once Phase 5 shows the confidence intervals separate.")

### What to conclude, honestly

Model D is only supported if **all** of the following hold:

1. It beats Model C on macro AUPRC, and
2. the intervals separate under patient-level bootstrap (Phase 5), across seeds, and
3. the gate demonstrably moved off identity and varies with the clinical input, and
4. the gains concentrate where the mechanism predicts (fluid biomarkers), not uniformly.

If (1) holds but (3) does not, the honest reading is that the extra parameters helped, not the
gating mechanism — and the paper claim has to be written that way.

If none hold, that is a clean negative result: with one CST value shared across all 49 B-scans of
a visit, clinical features may simply have no conditional signal left once the image is present.
The EDA flagged that ceiling in advance, which is what makes the negative result credible rather
than a failed experiment.

**Next:** `04_uncertainty_calibration.ipynb`.